# 01 — Pipeline local GRACE B2 đầy đủ

Chạy bằng kernel Python trong WSL. Tự tải Devign và ReVeal theo revision trong contract, trích lại Joern cho cả train/validation/test, tạo CodeT5 embedding, retrieval, prompt và ZIP cho validation/test. Không giới hạn số mẫu và không gọi LLM.

Quá trình có thể kéo dài nhiều giờ hoặc ngày. Joern checkpoint theo từng mẫu; giữ thư mục output để tiếp tục nếu bị ngắt. Mẫu Joern lỗi được giữ với graph_missing và báo rõ, không loại khỏi tập. Chỉ dùng cho nghiên cứu sau khi kiểm toán nhãn, trùng lặp và CPG.


In [ ]:
from pathlib import Path
import subprocess, sys, os
assert os.name != 'nt', 'Chọn kernel Python trong WSL, không dùng kernel Windows'
ROOT = Path('/mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified')
JOERN_HOME = Path('/home/yungn/tools/joern/y/joern-cli')
DATASETS = ['devign', 'reveal']
SPLITS = ['validation', 'test']
WORK = ROOT / 'output' / 'local_full'
for executable in ('joern-parse', 'joern-export'):
    file = JOERN_HOME / executable
    assert file.is_file() and os.access(file, os.X_OK), f'Joern không khả dụng: {file}'
assert (ROOT / 'evaluation_contract.yaml').is_file()
print('Root:', ROOT, 'Joern:', JOERN_HOME)


## 1. Tải dữ liệu đúng contract

Giữ các revision và split trong evaluation_contract.yaml. File raw có ID, code, nhãn và hash; không cần tự chuẩn bị JSONL.


In [ ]:
for dataset in DATASETS:
    raw = WORK / dataset / 'raw'
    subprocess.run([sys.executable, str(ROOT / 'download_contract_data.py'), '--dataset', dataset, '--out', str(raw)], cwd=ROOT, check=True)


## 2. Trích lại CPG bằng Joern

Trích toàn bộ train/validation/test; file CPG và manifest lưu trong WORK. Nếu bị ngắt, chạy lại cell để resume.


In [ ]:
for dataset in DATASETS:
    raw = WORK / dataset / 'raw'
    cpg = WORK / dataset / 'cpg'
    for split in ('train', 'validation', 'test'):
        subprocess.run([sys.executable, str(ROOT / 'prepare_joern_cpg.py'), '--input', str(raw / f'{split}.jsonl'), '--output', str(cpg / f'{split}.jsonl'), '--dataset', dataset, '--joern-home', str(JOERN_HOME)], cwd=ROOT, check=True)


## 3. Embedding, retrieval và đóng gói toàn bộ prompt

CodeT5 và tokenizer Qwen tải theo revision trong contract. Không dùng LIMIT; tạo ZIP đầy đủ cho cả validation và test. Kiểm tra tỷ lệ graph thiếu trước khi coi output là dữ liệu xác nhận.


In [ ]:
for dataset in DATASETS:
    cpg = WORK / dataset / 'cpg'
    for split in SPLITS:
        out = WORK / dataset / 'bundles' / split
        cmd = [sys.executable, str(ROOT / 'prepare_kaggle_bundle.py'), '--dataset', dataset, '--split', split, '--train', str(cpg / 'train.jsonl'), '--validation', str(cpg / 'validation.jsonl'), '--test', str(cpg / 'test.jsonl'), '--out', str(out), '--allow-missing-graphs']
        subprocess.run(cmd, cwd=ROOT, check=True)
        print('Kaggle bundle:', out / f'{dataset}_{split}_b2_kaggle.zip', flush=True)


Các ZIP nằm tại output/local_full/<dataset>/bundles/<split>/. Upload bundle tương ứng thành Kaggle Dataset riêng, rồi mở notebook 02. Notebook 02 vẫn mặc định smoke 20 mẫu.
